### Etapa 1 — Validar ambiente e arquivos

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Raiz do repositório
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

print("Python:", sys.executable)
print("Raiz do projeto:", ROOT)
print("\nArquivos de entrada:")

for nome in ["fortigate_syslog.log", "ICMP_Data_exfiltration.log.gz"]:
    arquivo = DATA / nome
    print(f"{nome}: {'OK' if arquivo.exists() else 'NÃO ENCONTRADO'}")
    if arquivo.exists():
        print(f"  Tamanho: {arquivo.stat().st_size:,} bytes")

print("\nPandas:", pd.__version__)

Python: c:\Users\marce\GitHub\network-anomaly-detection-lab\.venv\Scripts\python.exe
Raiz do projeto: c:\Users\marce\GitHub\network-anomaly-detection-lab

Arquivos de entrada:
fortigate_syslog.log: OK
  Tamanho: 129,118,208 bytes
ICMP_Data_exfiltration.log.gz: OK
  Tamanho: 4,391,546 bytes

Pandas: 3.1.1


### Etapa 2 — Inspecionar o formato dos logs

In [2]:
import gzip

arquivos = {
    "FortiGate": DATA / "fortigate_syslog.log",
    "ICMP comprimido": DATA / "ICMP_Data_exfiltration.log.gz",
}

for nome, caminho in arquivos.items():
    print(f"\n=== {nome} ===")

    abrir = gzip.open if caminho.suffix == ".gz" else open

    with abrir(caminho, "rt", encoding="utf-8", errors="replace") as f:
        for i, linha in enumerate(f, start=1):
            print(linha.rstrip()[:300])
            if i == 3:
                break


=== FortiGate ===
<190>date=2025-09-06 time=13:14:56 devname="FortiGate-VM64-KVM" devid="FGVMEV8QVPSH2E74" eventtime=1757175296503342540 tz="-0300" logid="0100044546" type="event" subtype="system" level="information" vd="root" logdesc="Attribute configured" user="admin" ui="jsconsole(192.168.15.6)" action="Edit" cfgt
<190>date=2025-09-06 time=13:17:05 devname="FortiGate-VM64-KVM" devid="FGVMEV8QVPSH2E74" eventtime=1757175426174358517 tz="-0300" logid="0101041986" type="event" subtype="vpn" level="information" vd="root" logdesc="Certificate regenerated" action="info" user="admin" ui="GUI(192.168.15.6)" name="Fort
<190>date=2025-09-06 time=13:17:13 devname="FortiGate-VM64-KVM" devid="FGVMEV8QVPSH2E74" eventtime=1757175433366110253 tz="-0300" logid="0101041986" type="event" subtype="vpn" level="information" vd="root" logdesc="Certificate regenerated" action="info" user="admin" ui="GUI(192.168.15.6)" name="Fort

=== ICMP comprimido ===
<190>date=2025-09-06 time=13:14:56 devname="FortiGate

In [3]:
import hashlib
import gzip

def hash_conteudo(caminho):
    abrir = gzip.open if caminho.suffix == ".gz" else open
    sha256 = hashlib.sha256()
    linhas = 0

    with abrir(caminho, "rb") as f:
        for linha in f:
            sha256.update(linha)
            linhas += 1

    return linhas, sha256.hexdigest()

for nome, caminho in arquivos.items():
    linhas, hash_sha = hash_conteudo(caminho)
    print(f"{nome}:")
    print(f"  Linhas: {linhas:,}")
    print(f"  SHA-256: {hash_sha}")

FortiGate:
  Linhas: 165,337
  SHA-256: 4e6b0b0ebde63a2ab5f51c062fe45fa329b535579c3338aefa21cf6072fb0b3c
ICMP comprimido:
  Linhas: 165,337
  SHA-256: 4e6b0b0ebde63a2ab5f51c062fe45fa329b535579c3338aefa21cf6072fb0b3c


### Etapa 3 — Mapear os eventos do FortiGate

In [4]:
import re
import pandas as pd

registros = []

with (DATA / "fortigate_syslog.log").open(
    encoding="utf-8", errors="replace"
) as arquivo:
    for linha in arquivo:
        campos = dict(
            re.findall(r'(\w+)=("[^"]*"|\S+)', linha)
        )
        campos = {
            chave: valor.strip('"')
            for chave, valor in campos.items()
        }
        registros.append(campos)

df = pd.DataFrame(registros)

print(f"Total de registros: {len(df):,}")
print(f"Colunas identificadas: {len(df.columns)}")

print("\n=== Tipos de log ===")
display(df["type"].value_counts(dropna=False).to_frame("quantidade"))

print("\n=== Subtipos ===")
display(df["subtype"].value_counts(dropna=False).head(15).to_frame("quantidade"))

print("\n=== Serviços mais frequentes ===")
display(df["service"].value_counts(dropna=False).head(15).to_frame("quantidade"))

print("\n=== Ações registradas ===")
display(df["action"].value_counts(dropna=False).head(15).to_frame("quantidade"))

Total de registros: 165,337
Colunas identificadas: 123

=== Tipos de log ===


,quantidade
type,
traffic,131726
utm,33289
event,296
Local,25
NaN,1



=== Subtipos ===


,quantidade
subtype,
forward,125670
app-ctrl,29291
local,6056
ssl,2770
dns,1034
system,279
webfilter,194
vpn,25
connector,16



=== Serviços mais frequentes ===


,quantidade
service,
PING,131169
HTTPS,11924
DNS,9299
SSL,5347
icmp/0/8,2032
HTTP,1694
NaN,1360
udp/15600,1034
udp/443,692



=== Ações registradas ===


,quantidade
action,
start,61980
accept,57115
pass,29291
close,6709
deny,2244
ip-conn,2032
info,1556
resign-as-untrusted,1239
redirect,1034


### Etapa 4 — Validar o protocolo ICMP

In [5]:
print("=== Protocolos mais frequentes ===")
display(
    df["proto"].value_counts(dropna=False).head(15).to_frame("quantidade")
)

print("\n=== Serviços por protocolo ===")
display(pd.crosstab(df["service"], df["proto"]).head(20))

print("\n=== Campos disponíveis para análise ===")
campos = [
    "date", "time", "srcip", "dstip", "proto", "service",
    "action", "sessionid", "duration", "sentbyte", "rcvdbyte",
    "sentpkt", "rcvdpkt", "logid"
]
print([campo for campo in campos if campo in df.columns])


=== Protocolos mais frequentes ===


,quantidade
proto,
1,133201
6,19533
17,12277
NaN,326



=== Serviços por protocolo ===


proto,1,17,6
service,,,
DHCP,0,14,0
DNS,0,8852,447
FTP,0,0,12
HTTP,0,0,1694
HTTPS,0,0,11924
NTP,0,294,0
PING,131169,0,0
RDP,0,0,3
SAMBA,0,0,19



=== Campos disponíveis para análise ===
['date', 'time', 'srcip', 'dstip', 'proto', 'service', 'action', 'sessionid', 'duration', 'sentbyte', 'rcvdbyte', 'sentpkt', 'rcvdpkt', 'logid']


### Etapa 5 — Criar a base de sessões ICMP

In [6]:
# Filtrar tráfego ICMP aceito
icmp = df[
    (df["proto"].astype(str) == "1") &
    (df["action"] == "accept")
].copy()

# Converter campos numéricos
numericos = [
    "sessionid", "duration", "sentbyte",
    "rcvdbyte", "sentpkt", "rcvdpkt"
]

for coluna in numericos:
    if coluna in icmp.columns:
        icmp[coluna] = pd.to_numeric(
            icmp[coluna], errors="coerce"
        ).fillna(0)

# Ordenar e consolidar atualizações por sessão
chaves = ["srcip", "dstip", "sessionid"]

icmp = icmp.sort_values(
    ["eventtime"] if "eventtime" in icmp.columns else ["date", "time"]
    if "date" in icmp.columns and "time" in icmp.columns
    else ["sessionid"]
)

sessoes_icmp = (
    icmp.groupby(chaves, dropna=False, sort=False)
    .tail(1)
    .copy()
)

print("Registros ICMP aceitos:", len(icmp))
print("Sessões após consolidação:", len(sessoes_icmp))
print("Origens:", sessoes_icmp["srcip"].nunique())
print("Destinos:", sessoes_icmp["dstip"].nunique())

display(
    sessoes_icmp.sort_values("sentbyte", ascending=False)[
        ["srcip", "dstip", "sessionid", "duration",
         "sentbyte", "rcvdbyte", "sentpkt", "rcvdpkt"]
    ].head(10)
)


Registros ICMP aceitos: 51939
Sessões após consolidação: 51402
Origens: 13
Destinos: 30


,srcip,dstip,sessionid,duration,sentbyte,rcvdbyte,sentpkt,rcvdpkt
86754,172.19.15.6,192.168.15.8,11437,97,36334761,36206027,65527,65295
93335,172.19.15.6,192.168.15.8,13952,102,36315345,36218846,65492,65318
95983,172.19.15.6,192.168.15.8,14889,101,36309794,36086763,65482,65080
90514,172.19.15.6,192.168.15.8,12903,110,36241079,36094655,65358,65094
68831,172.19.15.6,192.168.15.8,4379,126,16596948,16583953,31383,31353
73696,172.19.15.6,192.168.15.8,6929,127,16595289,16592088,31380,31373
68736,172.19.15.6,192.168.15.8,4336,127,16577191,16558287,31346,31305
66984,172.19.15.6,192.168.15.8,3589,145,16572509,16518590,31333,31214
67101,172.19.15.6,192.168.15.8,3649,120,16568013,16469234,31328,31137
61555,172.19.15.6,192.168.15.8,1024,442,8644977,8644977,41326,41326


### Etapa 6 — Engenharia de atributos e Isolation Forest

In [7]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Criar atributos por sessão
features = sessoes_icmp[
    ["srcip", "dstip", "sessionid", "duration",
     "sentbyte", "rcvdbyte", "sentpkt", "rcvdpkt"]
].copy()

features["duration"] = features["duration"].clip(lower=1)

features["bytes_per_second"] = (
    features["sentbyte"] / features["duration"]
)

features["packets_per_second"] = (
    features["sentpkt"] / features["duration"]
)

features["bytes_per_packet"] = (
    features["sentbyte"] /
    features["sentpkt"].replace(0, np.nan)
).fillna(0)

features["byte_ratio"] = (
    features["sentbyte"] /
    features["rcvdbyte"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Variáveis numéricas usadas pelo modelo
colunas_modelo = [
    "duration", "sentbyte", "rcvdbyte",
    "sentpkt", "rcvdpkt", "bytes_per_second",
    "packets_per_second", "bytes_per_packet", "byte_ratio"
]

X = features[colunas_modelo].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

modelo = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42
)

features["anomaly_label"] = modelo.fit_predict(X_scaled)
features["anomaly_score"] = -modelo.decision_function(X_scaled)
features["is_anomaly"] = features["anomaly_label"] == -1

print("Sessões analisadas:", len(features))
print("Anomalias identificadas:", features["is_anomaly"].sum())

display(
    features.sort_values("anomaly_score", ascending=False)[
        ["srcip", "dstip", "sessionid", "sentbyte", "duration",
         "bytes_per_second", "packets_per_second",
         "anomaly_score", "is_anomaly"]
    ].head(20)
)

Sessões analisadas: 51402
Anomalias identificadas: 391


,srcip,dstip,sessionid,sentbyte,duration,bytes_per_second,packets_per_second,anomaly_score,is_anomaly
22555,172.19.15.5,7.7.7.7,13360,84,65,1.292308,0.015385,0.046555,True
10362,172.19.15.5,7.7.7.7,7105,84,65,1.292308,0.015385,0.046555,True
98680,172.19.15.8,2.2.2.2,15934,84,65,1.292308,0.015385,0.046555,True
98852,172.19.15.10,3.3.3.3,16014,84,65,1.292308,0.015385,0.046555,True
42067,172.19.15.5,7.7.7.7,23234,84,65,1.292308,0.015385,0.046555,True
19230,172.19.15.5,7.7.7.7,11662,84,65,1.292308,0.015385,0.046555,True
3930,172.19.15.5,7.7.7.7,3844,84,65,1.292308,0.015385,0.046555,True
41086,172.19.15.5,7.7.7.7,22679,84,65,1.292308,0.015385,0.046555,True
119524,172.19.15.8,2.2.2.2,27732,84,65,1.292308,0.015385,0.046555,True
119222,172.19.15.9,10.1.1.1,27612,84,65,1.292308,0.015385,0.046555,True


In [8]:
# 1. Maiores sessões por volume enviado
print("MAIORES SESSÕES ICMP POR VOLUME ENVIADO")

display(
    features.sort_values("sentbyte", ascending=False)[
        [
            "srcip", "dstip", "sessionid",
            "sentbyte", "rcvdbyte", "duration",
            "bytes_per_second", "anomaly_score", "is_anomaly"
        ]
    ].head(15)
)

# 2. Anomalias por volume enviado
print("ANOMALIAS COM MAIOR VOLUME ENVIADO")

display(
    features[features["is_anomaly"]]
    .sort_values("sentbyte", ascending=False)[
        [
            "srcip", "dstip", "sessionid",
            "sentbyte", "rcvdbyte", "duration",
            "bytes_per_second", "anomaly_score"
        ]
    ].head(15)
)

MAIORES SESSÕES ICMP POR VOLUME ENVIADO


,srcip,dstip,sessionid,sentbyte,rcvdbyte,duration,bytes_per_second,anomaly_score,is_anomaly
86754,172.19.15.6,192.168.15.8,11437,36334761,36206027,97,374585.164948,-0.098786,False
93335,172.19.15.6,192.168.15.8,13952,36315345,36218846,102,356032.794118,-0.096884,False
95983,172.19.15.6,192.168.15.8,14889,36309794,36086763,101,359502.910891,-0.098786,False
90514,172.19.15.6,192.168.15.8,12903,36241079,36094655,110,329464.354545,-0.096688,False
68831,172.19.15.6,192.168.15.8,4379,16596948,16583953,126,131721.809524,-0.100672,False
73696,172.19.15.6,192.168.15.8,6929,16595289,16592088,127,130671.566929,-0.102450,False
68736,172.19.15.6,192.168.15.8,4336,16577191,16558287,127,130529.062992,-0.100672,False
66984,172.19.15.6,192.168.15.8,3589,16572509,16518590,145,114293.165517,-0.100672,False
67101,172.19.15.6,192.168.15.8,3649,16568013,16469234,120,138066.775000,-0.100672,False
61555,172.19.15.6,192.168.15.8,1024,8644977,8644977,442,19558.771493,-0.107352,False


ANOMALIAS COM MAIOR VOLUME ENVIADO


,srcip,dstip,sessionid,sentbyte,rcvdbyte,duration,bytes_per_second,anomaly_score
523,172.19.15.5,7.7.7.7,2167,84,0,65,1.292308,0.046555
882,172.19.15.5,7.7.7.7,2275,84,0,65,1.292308,0.046555
1323,172.19.15.5,7.7.7.7,2482,84,0,65,1.292308,0.046555
1586,172.19.15.5,7.7.7.7,2613,84,0,65,1.292308,0.046555
2270,172.19.15.5,7.7.7.7,2976,84,0,65,1.292308,0.046555
2512,172.19.15.5,7.7.7.7,3111,84,0,65,1.292308,0.046555
2984,172.19.15.5,7.7.7.7,3319,84,0,65,1.292308,0.046555
3259,172.19.15.5,7.7.7.7,3480,84,0,65,1.292308,0.046555
3496,172.19.15.5,7.7.7.7,3629,84,0,65,1.292308,0.046555
3930,172.19.15.5,7.7.7.7,3844,84,0,65,1.292308,0.046555


### Etapa 7 - Validar as sessões suspeitas

In [9]:
filtro = (
    (df["proto"].astype(str) == "1")
    & (df["srcip"] == "172.19.15.6")
    & (df["dstip"] == "192.168.15.8")
)

registros = df.loc[filtro].copy()

print("Registros encontrados:", len(registros))
print("Ações:", registros["action"].value_counts(dropna=False).to_dict())
print("Log IDs:", registros["logid"].value_counts(dropna=False).to_dict())

display(
    registros[
        [
            c for c in [
                "date", "time", "srcip", "dstip", "sessionid",
                "action", "logid", "duration", "sentbyte",
                "rcvdbyte", "sentpkt", "rcvdpkt", "identifier"
            ] if c in registros.columns
        ]
    ].head(20)
)

Registros encontrados: 36
Ações: {'accept': 15, 'start': 12, 'pass': 7, 'ip-conn': 2}
Log IDs: {'0000000015': 12, '0000000013': 12, '1059028704': 7, '0000000020': 3, '0000000011': 2}


,date,time,srcip,dstip,sessionid,action,logid,duration,sentbyte,rcvdbyte,sentpkt,rcvdpkt,identifier
55643,2025-09-06,16:21:53,172.19.15.6,192.168.15.8,30530,start,0000000015,0,0,0,0,0,0
56246,2025-09-06,16:23:39,172.19.15.6,192.168.15.8,30530,accept,0000000013,105,84,84,2,2,0
61403,2025-09-06,17:40:54,172.19.15.6,192.168.15.8,1024,start,0000000015,0,0,0,0,0,0
61447,2025-09-06,17:42:56,172.19.15.6,192.168.15.8,1024,accept,0000000020,121,2705111,2705111,12950,12950,0
61485,2025-09-06,17:44:56,172.19.15.6,192.168.15.8,1024,accept,0000000020,242,5418499,5418499,25937,25937,0
61509,2025-09-06,17:46:59,172.19.15.6,192.168.15.8,1024,accept,0000000020,364,8279976,8279976,39585,39585,0
61554,2025-09-06,17:48:17,172.19.15.6,192.168.15.8,1024,ip-conn,0000000011,NaN,NaN,NaN,NaN,NaN,0
61555,2025-09-06,17:48:17,172.19.15.6,192.168.15.8,1024,accept,0000000013,442,8644977,8644977,41326,41326,0
62412,2025-09-06,17:55:09,172.19.15.6,192.168.15.8,1527,start,0000000015,0,0,0,0,0,0
62413,2025-09-06,17:55:09,172.19.15.6,192.168.15.8,1527,pass,1059028704,NaN,NaN,NaN,NaN,NaN,NaN


### Etapa 8 - Analisar todas as sessões desse par de IPs

In [10]:
filtro = (
    (df["proto"].astype(str) == "1")
    & (df["srcip"] == "172.19.15.6")
    & (df["dstip"] == "192.168.15.8")
    & (df["action"] == "accept")
)

detalhes = df.loc[filtro].copy()

for coluna in ["duration", "sentbyte", "rcvdbyte", "sentpkt", "rcvdpkt", "identifier"]:
    detalhes[coluna] = pd.to_numeric(detalhes[coluna], errors="coerce")

# Considera apenas o registro final de cada sessão
finais = (
    detalhes.sort_values(["date", "time"])
    .groupby("sessionid", dropna=False)
    .tail(1)
)

display(
    finais[
        [
            "date", "time", "sessionid", "logid",
            "duration", "sentbyte", "rcvdbyte",
            "sentpkt", "rcvdpkt", "identifier"
        ]
    ].sort_values("sentbyte", ascending=False)
)

,date,time,sessionid,logid,duration,sentbyte,rcvdbyte,sentpkt,rcvdpkt,identifier
86754,2025-09-06,22:33:21,11437,0000000013,97,36334761,36206027,65527,65295,5684
93335,2025-09-06,22:43:03,13952,0000000013,102,36315345,36218846,65492,65318,7196
95983,2025-09-06,22:47:08,14889,0000000013,101,36309794,36086763,65482,65080,3684
90514,2025-09-06,22:38:39,12903,0000000013,110,36241079,36094655,65358,65094,8568
68831,2025-09-06,19:28:45,4379,0000000013,126,16596948,16583953,31383,31353,8684
73696,2025-09-06,21:56:43,6929,0000000013,127,16595289,16592088,31380,31373,7640
68736,2025-09-06,19:21:19,4336,0000000013,127,16577191,16558287,31346,31305,6164
66984,2025-09-06,18:36:59,3589,0000000013,145,16572509,16518590,31333,31214,6776
67101,2025-09-06,18:39:54,3649,0000000013,120,16568013,16469234,31328,31137,9348
61555,2025-09-06,17:48:17,1024,0000000013,442,8644977,8644977,41326,41326,0


## Etapa 9 — Galeria de visualizações

As figuras abaixo são geradas no projeto e ficam na pasta `assets/`. Os caminhos são relativos a este notebook (`notebooks/`).

### 9.1 — Comparação das sessões prioritárias

Compara o volume enviado e a duração das nove sessões de maior volume identificadas no par investigado.

![Comparação de volume enviado e duração das sessões prioritárias](../assets/icmp_priority_sessions_comparison.png)

### 9.2 — Detalhe das sessões prioritárias

Mostra a distribuição de volume e duração das sessões selecionadas. Os rótulos próximos podem ficar visualmente densos; para leitura precisa, consulte também a tabela de sessões no relatório.

![Detalhe das sessões prioritárias](../assets/icmp_priority_sessions_detail.png)

### 9.3 — Volume por par origem-destino

Destaca os pares origem-destino com maior volume agregado. O volume agregado deve ser interpretado considerando que os contadores do firewall podem ser cumulativos e que a consolidação é aproximada.

![Top pares de origem e destino por volume](../assets/icmp_top_source_destination_pairs.png)

### 9.4 — Sessões com maior volume enviado

Ordena as sessões de maior volume enviado para apoiar a triagem. Valores altos são indicadores para investigação, não confirmação isolada de exfiltração.

![Sessões ICMP com maior volume enviado](../assets/icmp_top_sessions_volume.png)

### 9.5 — Volume enviado versus duração

Relaciona duração e volume enviado. A concentração de pontos e a diferença de escala podem dificultar a leitura; use este gráfico como visão exploratória e complemente com as comparações específicas.

![Volume enviado versus duração das sessões ICMP](../assets/icmp_volume_vs_duration.png)

### 9.6 — Linha do tempo do par suspeito

Apresenta os horários e volumes das sessões observadas entre `172.19.15.6` e `192.168.15.8` em 6 de setembro de 2025.

![Linha do tempo das sessões ICMP do par investigado](../assets/icmp_suspicious_pair_timeline.png)

> **Nota metodológica:** os gráficos representam os resultados da análise exploratória dos logs do FortiGate. O padrão observado é compatível com possível tunelamento ou transferência ICMP não usual, mas os logs, sem inspeção do payload e validação dos ativos, não comprovam exfiltração.
